# 02 — M1: LeNet từ số 0 (mốc tham chiếu)

**Chủ: Hoàng** · lý thuyết: `docs/LY_THUYET.md` phần 2 · code: `src/gtsrb/models/m1_lenet.py`

M1 **cố ý đơn giản**: không BatchNorm, không residual, không spatial dropout.
Nó tồn tại để làm **mốc** — mọi điểm accuracy mà M2 hơn M1 phải quy được về
một thành phần cụ thể. Vì vậy **không** được "tinh chỉnh M1 cho mạnh lên".

Train trước khi chạy notebook: `make train-m1`

In [ ]:
# Cho notebook thấy src/.  Chạy ô này đầu tiên.
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import os; os.chdir(ROOT)          # mọi đường dẫn trong repo là tương đối từ gốc

import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from gtsrb import CLASS_NAMES, NUM_CLASSES
print("torch", torch.__version__, "| MPS", torch.backends.mps.is_available(),
      "| CUDA", torch.cuda.is_available())

In [ ]:
# Tiện ích dùng chung: tìm và nạp lại một run đã train
import glob, json
sys.path.insert(0, str(ROOT / "scripts"))
from evaluate import load_run, test_loader_for
from gtsrb.utils.seed import pick_device

device = pick_device("auto")

def find_runs(prefix=""):
    """Danh sách thư mục run có checkpoint, lọc theo tiền tố tên model."""
    dirs = [pathlib.Path(p) for p in sorted(glob.glob("artifacts/runs/*"))]
    return [d for d in dirs if (d / "best.pt").exists() and d.name.startswith(prefix)]

def latest(prefix):
    runs = find_runs(prefix)
    if not runs:
        raise FileNotFoundError(
            f"Chưa có run nào tên bắt đầu bằng {prefix!r}. Chạy: make train-{prefix[:2]}")
    return runs[-1]

print("Các run đã có:")
for d in find_runs(): print("  ", d.name)

## 1. Kiến trúc và luồng tensor

```
đầu vào           (B, 3, 48, 48)
Conv(3→32, 5×5)   (B, 32, 48, 48)   pad=2 nên giữ kích thước
ReLU + MaxPool2   (B, 32, 24, 24)
Conv(32→64, 5×5)  (B, 64, 24, 24)
ReLU + MaxPool2   (B, 64, 12, 12)
Flatten           (B, 9216)          64 × 12 × 12 = 9216
Linear(9216→256)  (B, 256)           ← lớp "đắt" nhất
ReLU + Dropout
Linear(256→43)    (B, 43)
```

In [ ]:
from gtsrb.models.registry import build_model, count_parameters, parameter_table

m1 = build_model("m1_lenet", img_size=48)
print(m1)
print(f"\nTổng tham số: {count_parameters(m1):,}")

## 2. ★ Bài học chính của M1: tham số nằm ở đâu?

In [ ]:
table = parameter_table(m1)
print(table.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.barh(table["layer"], table["percent"], color="indianred")
ax.set_xlabel("% tổng tham số"); ax.invert_yaxis()
ax.set_title(f"M1: {table.iloc[0]['percent']:.1f}% tham số nằm ở MỘT lớp fully-connected")
plt.tight_layout(); plt.show()

**Kết quả đo thật:** `classifier.1` (Linear 9216→256) có **2.359.552** tham số
= **97,3%** của tổng **2.424.299**.

Hai lớp conv cộng lại chỉ ~2,2%.

→ Đây chính là lý do mọi kiến trúc hiện đại (ResNet, MobileNet, EfficientNet, và M2
của nhóm) **bỏ Flatten+FC và dùng Global Average Pooling**. M2 thay chỗ này bằng
`GAP → FC(256→43)` chỉ **11.051** tham số — giảm ~213 lần.

**Câu giảng viên sẽ hỏi:** *"M2 sâu gấp 4 lần M1, vậy nó nhiều tham số hơn bao nhiêu?"*
→ **Nó ÍT HƠN**: 1.237.451 so với 2.424.299. Vì conv dùng tham số chia sẻ (một kernel
3×3 trượt khắp ảnh) nên rất rẻ, còn chỗ đắt là FC thì M2 đã bỏ.

## 3. Đường cong huấn luyện

In [ ]:
run = latest("m1_lenet")
history = pd.read_csv(run / "train_log.csv")
print("run:", run.name, "| số epoch:", len(history))

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
axes[0].plot(history.epoch, history.train_loss, label="train loss")
axes[0].set_title("Loss"); axes[0].set_xlabel("epoch"); axes[0].legend(); axes[0].grid(alpha=.3)

axes[1].plot(history.epoch, history.train_acc, label="train acc")
axes[1].plot(history.epoch, history.val_top1, label="val top-1")
axes[1].set_title("Accuracy"); axes[1].set_xlabel("epoch"); axes[1].legend(); axes[1].grid(alpha=.3)

axes[2].plot(history.epoch, history.val_macro_f1, color="darkgreen", label="val macro-F1")
best = history.loc[history.val_macro_f1.idxmax()]
axes[2].scatter([best.epoch], [best.val_macro_f1], c="red", zorder=5,
                label=f"best = {best.val_macro_f1:.4f} @ epoch {int(best.epoch)}")
axes[2].set_title("macro-F1 — chỉ số dùng cho early stopping")
axes[2].set_xlabel("epoch"); axes[2].legend(); axes[2].grid(alpha=.3)
plt.tight_layout(); plt.show()

**Vì sao early stopping theo macro-F1 chứ không theo accuracy?**

Dữ liệu mất cân bằng 10,7:1. Accuracy bị lớp đông chi phối: model có thể bỏ hẳn một
lớp 210 ảnh (0,54% dữ liệu) mà accuracy chỉ giảm ~0,54 điểm — gần như không thấy.
Macro-F1 lấy trung bình F1 của **cả 43 lớp với trọng số bằng nhau** nên mất ~2,3 điểm.

Nguyên tắc: **chọn checkpoint theo đúng chỉ số mình thật sự quan tâm.**

## 4. Learning rate: warmup + cosine

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(history.epoch, history.lr, marker="o", ms=3)
ax.set_xlabel("epoch"); ax.set_ylabel("learning rate")
ax.set_title("Warmup tuyến tính 3 epoch rồi cosine annealing")
ax.grid(alpha=.3); plt.tight_layout(); plt.show()

print("Vì sao warmup: ở những bước đầu, ước lượng moment bậc 2 (v_hat) của Adam")
print("được tính từ RẤT ÍT mẫu nên rất nhiễu -> bước đi có thể rất lớn và sai hướng.")
print()
print("Vì sao cosine: đầu kỳ LR lớn để khám phá rộng, cuối kỳ LR -> gần 0 để lắng")
print("vào đáy. So với step decay: không phải đoán 'giảm ở epoch nào'.")

## 5. Đánh giá trên tập test

Tập test **chỉ mở một lần, ở cuối**. Mọi quyết định trước đó chỉ dựa vào val.

In [ ]:
from gtsrb.eval.metrics import evaluate, worst_classes

model, cfg, _ = load_run(run, device)
loader = test_loader_for(cfg)
stats = evaluate(model, loader, device)

print(f"top-1       = {stats['top1']:.4f}")
print(f"top-5       = {stats['top5']:.4f}   <- bão hoà trên 43 lớp, chỉ số YẾU")
print(f"macro-F1    = {stats['macro_f1']:.4f}   <- CHỈ SỐ CHÍNH")
print(f"weighted-F1 = {stats['weighted_f1']:.4f}")
print()
print("10 lớp yếu nhất:")
print(worst_classes(stats["per_class"], 10).to_string(index=False))

## 6. Checklist Hoàng phải trả lời được

- Vì sao hai conv 3×3 thay một conv 5×5? → 18C² vs 25C², và **2** tầng phi tuyến
- BatchNorm làm gì, đặt ở đâu? → Conv→BN→ReLU; tác dụng thật là làm mượt loss landscape
- Residual giải quyết vấn đề gì? → **degradation**, không phải overfitting; `∂y/∂x = ∂F/∂x + 1`
- SpatialDropout khác Dropout thường? → bỏ **cả kênh**, vì pixel lân cận tương quan cao
- GAP thay Flatten+FC để làm gì? → M1 là bằng chứng: 97,3% tham số ở một lớp FC
- Label smoothing công thức? → `(1−ε)y + ε/K`; K=43, ε=0,1 → 0,9023 / 0,00233
- Early stopping theo macro-F1 vì sao? → mất cân bằng 10,7:1

Đáp án đầy đủ: `docs/PHAN_CONG.md` mục **HOÀNG**, 10 câu ở cuối.